# DiffCRL on Google Colab

This notebook runs the repository's existing `train_diffcrl.py` entry point without changing its scientific configuration. It keeps credentials out of clone URLs and notebook output, installs the custom MetaWorld checkout separately, reads SAC experts from Drive, and writes runs back to Drive. Formal runs are guarded and never start during setup.

Before starting, select a GPU runtime if desired. Store each SAC `.zip` expert **and its manifest sidecar** (`.zip.manifest.json`, `.manifest.json`, or `expert_manifest.json`) under the configured expert directory. KUKA-v3 formal experiments require manifests; the model files are not expected to be in Git.

In [1]:
# Mount persistent storage first.
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ---- User-editable paths and repository settings ----
from pathlib import Path

GITHUB_REPOSITORY = 'yikai-he/continual-learning'
GITHUB_REF = '57b8c38'  # Recommended: an immutable commit SHA; blank uses the default branch.
REPO_DIR = Path('/content/continual-learning')

# KUKA-v3 uses the project's custom MetaWorld fork, obtained separately.
KUKA_METAWORLD_REPOSITORY = 'YOUR_GITHUB_USER/metaworld-kuka-v3'
KUKA_METAWORLD_REF = ''  # Recommended: an immutable commit SHA.
KUKA_METAWORLD_DIR = Path('/content/metaworld-kuka-v3')
USE_KUKA_V3 = True

DRIVE_ROOT = Path('/content/drive/MyDrive/DiffCRL')
SAC_EXPERT_DIR = DRIVE_ROOT / 'experts'
DRIVE_EXPERIMENT_DIR = DRIVE_ROOT / 'experiments'
LOCAL_EXPERIMENT_DIR = Path('/content/experiments')
STORAGE_MODE = 'local-sync'  # Recommended: 'local-sync' or legacy 'direct-drive'.
EXPERIMENT_OUTPUT_DIR = (LOCAL_EXPERIMENT_DIR if STORAGE_MODE == 'local-sync'
                         else DRIVE_EXPERIMENT_DIR)
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints'
EXISTING_EXPERIMENT_DATA_DIR = DRIVE_ROOT / 'existing_experiment_data'
GENERATED_CONFIG_DIR = Path('/content/diffcrl_colab_configs')

for path in (SAC_EXPERT_DIR, EXPERIMENT_OUTPUT_DIR, DRIVE_EXPERIMENT_DIR, CHECKPOINT_DIR,
             EXISTING_EXPERIMENT_DATA_DIR, GENERATED_CONFIG_DIR):
    path.mkdir(parents=True, exist_ok=True)
print('Drive root:', DRIVE_ROOT)

Drive root: /content/drive/MyDrive/DiffCRL


## Secure private-repository clone

Put a fine-grained, read-only GitHub token in Colab Secrets as `GITHUB_TOKEN` (preferred), or enter it at the hidden prompt. The token is passed through a temporary `GIT_ASKPASS` helper: it is not embedded in the URL, printed, or persisted in Git configuration. Revoke tokens that may have been exposed elsewhere.

In [ ]:
import getpass, os, stat, subprocess, tempfile

def secret(name):
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = None
    return value or getpass.getpass(f'{name} (hidden): ')

def clone_private(repository, destination, ref='', token=None):
    destination = Path(destination)
    if (destination / '.git').is_dir():
        print(f'Using existing checkout: {destination}')
        return
    if destination.exists() and any(destination.iterdir()):
        raise FileExistsError(f'Refusing non-empty clone destination: {destination}')
    token = token or secret('GITHUB_TOKEN')
    with tempfile.TemporaryDirectory() as temp_dir:
        askpass = Path(temp_dir) / 'askpass.sh'
        askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token;; *) printf %s "$GITHUB_TOKEN";; esac\n')
        askpass.chmod(stat.S_IRUSR | stat.S_IWUSR | stat.S_IXUSR)
        env = os.environ.copy()
        env.update(GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT='0', GITHUB_TOKEN=token)
        subprocess.run(['git', 'clone', '--filter=blob:none',
                        f'https://github.com/{repository}.git', str(destination)],
                       check=True, env=env)
    if ref:
        subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', ref], check=True)

github_token = secret('GITHUB_TOKEN')
clone_private(GITHUB_REPOSITORY, REPO_DIR, GITHUB_REF, github_token)
if USE_KUKA_V3:
    clone_private(KUKA_METAWORLD_REPOSITORY, KUKA_METAWORLD_DIR,
                  KUKA_METAWORLD_REF, github_token)
del github_token
print(subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip())

## Reproducible dependencies

`requirements-colab.txt` pins the Python dependencies used by the experiment. Colab's preinstalled PyTorch is retained because it is matched to the runtime CUDA image; its exact version is recorded below. Installing the KUKA fork after the public requirements intentionally replaces the public `metaworld` wheel while retaining the repository's v3 public interface. Restart the runtime if pip reports that an already-imported package changed.

In [ ]:
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', str(REPO_DIR / 'requirements-colab.txt')], check=True)
if USE_KUKA_V3:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    '--no-deps', '-e', str(KUKA_METAWORLD_DIR)], check=True)

In [ ]:
# CUDA and environment check (training still supports CPU).
import importlib.metadata as md, platform, torch
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA build:', torch.version.cuda)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
for package in ('metaworld', 'gymnasium', 'stable-baselines3', 'mujoco', 'PyYAML'):
    print(f'{package}: {md.version(package)}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
assert DEVICE in ('cpu', 'cuda')

## Expert files and path-only configuration

Edit the mapping to match the task names in the selected YAML. Relative values resolve below `SAC_EXPERT_DIR`; absolute paths are also accepted. Sidecar manifests must remain beside their model. The helper copies the selected repository YAML and modifies only expert paths, output path, device, and an optional collision-safe run name.

In [ ]:
import yaml

EXPERT_FILES = {
    'kuka-reach-v3': 'reach-v3/expert_model.zip',
    'kuka-handle-press-side-v3': 'handle-press-side-v3/expert_model.zip',
    'kuka-drawer-close-v3': 'drawer-close-v3/expert_model.zip',
    'kuka-faucet-close-v3': 'faucet-close-v3/expert_model.zip',
    'kuka-hammer-v3': 'hammer-v3/expert_model.zip',
}

def expert_path(value):
    value = Path(value)
    return value if value.is_absolute() else SAC_EXPERT_DIR / value

def make_colab_config(source_relative, name, run_name=None):
    source = REPO_DIR / source_relative
    config = yaml.safe_load(source.read_text())
    tasks = config['continual']['tasks']
    missing_mapping = [task for task in tasks if task not in EXPERT_FILES]
    if missing_mapping:
        raise KeyError(f'Add expert paths for: {missing_mapping}')
    config['continual']['experts'] = {task: str(expert_path(EXPERT_FILES[task]))
                                      for task in tasks}
    config['runtime']['output'] = str(EXPERIMENT_OUTPUT_DIR / name)
    config['runtime']['device'] = DEVICE
    config['runtime']['run_name'] = run_name
    destination = GENERATED_CONFIG_DIR / f'{name}.yaml'
    destination.write_text(yaml.safe_dump(config, sort_keys=False))
    return destination, config

def verify_experts(config):
    missing = []
    for task, model in config['continual']['experts'].items():
        model = Path(model)
        manifests = (model.with_suffix(model.suffix + '.manifest.json'),
                     model.with_suffix('.manifest.json'), model.parent / 'expert_manifest.json')
        if not model.is_file(): missing.append(f'{task}: model {model}')
        if config.get('environment', {}).get('backend') == 'kuka-v3' and not any(p.is_file() for p in manifests):
            missing.append(f'{task}: manifest beside {model}')
    if missing:
        raise FileNotFoundError('Missing required expert artifacts:\n' + '\n'.join(missing))
    print(f'Validated {len(config["continual"]["experts"])} expert paths.')

## Lightweight checks

The first cell performs imports, CLI parsing, configuration validation, expert validation, and one environment reset. It does not train. The optional smoke run uses the repository's existing two-task smoke configuration (4 trajectories/task, 2 diffusion epochs, 2 BC epochs, and 2 evaluation episodes); it is small but still performs environment rollouts.

In [ ]:
# Fast non-training smoke check.
import os, subprocess, sys
os.chdir(REPO_DIR)
SMOKE_SOURCE = ('configs/diffcrl/kuka_diffcrl_smoke.yaml' if USE_KUKA_V3
                else 'configs/diffcrl/diffcrl_smoke.yaml')
smoke_path, smoke_config = make_colab_config(SMOKE_SOURCE, 'smoke', run_name='colab-smoke')
verify_experts(smoke_config)
subprocess.run([sys.executable, 'train_diffcrl.py', '--help'], check=True)
from src.config import load_config
from src.envs import EnvironmentSettings
cfg = load_config(smoke_path, expected_experiment='diffcrl')
env = EnvironmentSettings.from_config(cfg.environment).make(cfg.continual.tasks[0], cfg.runtime.seed)
observation, info = env.reset(seed=cfg.runtime.seed)
print('Reset OK:', cfg.continual.tasks[0], observation.shape, env.action_space.shape)
env.close()

In [ ]:
RUN_SMOKE_TEST = False  # Explicitly opt in.
if RUN_SMOKE_TEST:
    subprocess.run([sys.executable, '-u', 'train_diffcrl.py',
                    '--config', str(smoke_path)], cwd=REPO_DIR, check=True)
else:
    print('Smoke training skipped; set RUN_SMOKE_TEST=True to run it.')

## Formal DiffCRL and No-Replay baseline

These cells use the maintained repository configurations unchanged except for runtime paths/device. Choose a new `RUN_NAME` for every attempt: the trainer intentionally refuses to overwrite an existing run. In recommended `local-sync` mode training uses `/content/experiments` and each validated stage is automatically copied to Drive; `direct-drive` preserves the older workflow.

In [ ]:
RUN_NAME = 'seed0-colab'  # Change for each independent run.
FORMAL_SOURCE = ('configs/diffcrl/kuka_diffcrl_5task.yaml' if USE_KUKA_V3
                 else 'configs/diffcrl/diffcrl.yaml')
NO_REPLAY_SOURCE = ('configs/diffcrl/kuka_diffcrl_no_replay_5task.yaml' if USE_KUKA_V3
                    else 'configs/diffcrl/diffcrl_no_replay.yaml')
formal_path, formal_config = make_colab_config(FORMAL_SOURCE, 'formal_diffcrl', RUN_NAME)
baseline_path, baseline_config = make_colab_config(NO_REPLAY_SOURCE, 'formal_no_replay', RUN_NAME)
verify_experts(formal_config)
verify_experts(baseline_config)
print(formal_path, baseline_path, sep='\n')

In [ ]:
RUN_FORMAL_DIFFCRL = False  # Expensive: review generated YAML before enabling.
if RUN_FORMAL_DIFFCRL:
    command = [sys.executable, '-u', 'train_diffcrl.py', '--config', str(formal_path)]
    if STORAGE_MODE == 'local-sync':
        command += ['--stage-sync-to', str(DRIVE_EXPERIMENT_DIR / 'formal_diffcrl' / RUN_NAME)]
    subprocess.run(command, cwd=REPO_DIR, check=True)
else:
    print('Formal DiffCRL skipped.')

In [ ]:
RUN_FORMAL_NO_REPLAY = False  # Expensive: review generated YAML before enabling.
if RUN_FORMAL_NO_REPLAY:
    subprocess.run([sys.executable, '-u', 'train_diffcrl.py',
                    '--config', str(baseline_path)], cwd=REPO_DIR, check=True)
else:
    print('Formal No-Replay baseline skipped.')

## Checkpoint and result archive

The run itself already lives under `EXPERIMENT_OUTPUT_DIR` on Drive. The helper below optionally creates a second checkpoint-focused archive. It never overwrites an existing archive. `EXISTING_EXPERIMENT_DATA_DIR` is reserved for prior run directories or other data you upload; the current trainer does not consume it automatically.

In [ ]:
import shutil

def archive_run(run_dir, archive_name):
    run_dir, destination = Path(run_dir), CHECKPOINT_DIR / archive_name
    if not run_dir.is_dir(): raise FileNotFoundError(run_dir)
    if destination.exists(): raise FileExistsError(destination)
    destination.mkdir(parents=True)
    patterns = ('*.pt', '*.yaml', '*.json', 'RUNNING', 'RUN_COMPLETE')
    copied = []
    for pattern in patterns:
        for source in run_dir.rglob(pattern):
            target = destination / source.relative_to(run_dir)
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, target); copied.append(target)
    print(f'Archived {len(copied)} files to {destination}')
    return destination

print('Prior experiment data:', EXISTING_EXPERIMENT_DATA_DIR)
print('Entries:', [p.name for p in EXISTING_EXPERIMENT_DATA_DIR.iterdir()])
# Example after completion:
# archive_run(EXPERIMENT_OUTPUT_DIR / 'formal_diffcrl' / RUN_NAME, f'formal_diffcrl-{RUN_NAME}')

## Checkpoint/resume behavior

DiffCRL writes `stage_<index>_COMPLETE.json` only after a stage's checkpoints, evaluation matrix, report, expert-integrity checks, and boundary state are complete. In local-sync mode the Drive marker is published last, after destination SHA-256 verification. Resume validates every artifact hash, restores policy/diffusion and normalization state, replay metadata, evaluation history, and RNG state, then skips completed stages. Partial-stage artifacts are preserved under their original names; a retry uses a `.resume_N` attempt path.

For direct-drive mode, resume the Drive directory in place. For local-sync mode, validate and restore the Drive backup to a fresh `/content` directory first, then resume that local copy while keeping `--stage-sync-to` enabled. The run directory and expert paths may move, but scientific settings and expert bytes must match. Runs created before completion markers are not resumable through this interface. Exact numeric equivalence is not guaranteed if the hardware/device backend changes.

In [ ]:
# Optional stage-boundary resume from persistent Google Drive storage.
RESUME_RUN_DIR = None  # Example: EXPERIMENT_OUTPUT_DIR / 'formal_diffcrl' / RUN_NAME
RESUME_CONFIG = formal_path  # Or baseline_path for a No-Replay run.
RUN_RESUME = False
if RUN_RESUME:
    if RESUME_RUN_DIR is None:
        raise ValueError('Set RESUME_RUN_DIR to an existing Drive run directory.')
    resume_dir = Path(RESUME_RUN_DIR)
    if not resume_dir.is_dir():
        raise FileNotFoundError(resume_dir)
    markers = sorted(resume_dir.glob('stage_*_COMPLETE.json'))
    print('Completion markers:', [path.name for path in markers])
    subprocess.run([sys.executable, '-u', 'train_diffcrl.py',
                    '--config', str(RESUME_CONFIG),
                    '--resume-from', str(resume_dir)],
                   cwd=REPO_DIR, check=True)
else:
    print('Resume skipped; set a Drive path and RUN_RESUME=True to continue.')

## Validate, restore, retry, and inspect a local-sync backup

These commands operate only on manifest-referenced resume artifacts. A failed sync can be retried without retraining. Restore refuses a non-empty destination and ignores incomplete remote stages because only validated completion markers define the restorable prefix.

In [ ]:
DRIVE_BACKUP_RUN = DRIVE_EXPERIMENT_DIR / 'formal_diffcrl' / RUN_NAME
LOCAL_RESTORED_RUN = LOCAL_EXPERIMENT_DIR / 'formal_diffcrl' / RUN_NAME
SYNC_ACTION = None  # Choose: 'status', 'validate', 'restore', or 'sync'.
if SYNC_ACTION:
    command = [sys.executable, 'manage_diffcrl_sync.py', SYNC_ACTION,
               '--source', str(DRIVE_BACKUP_RUN)]
    if SYNC_ACTION == 'restore':
        command += ['--destination', str(LOCAL_RESTORED_RUN)]
    elif SYNC_ACTION == 'sync':  # Retry from a surviving local run.
        command = [sys.executable, 'manage_diffcrl_sync.py', 'sync',
                   '--source', str(LOCAL_RESTORED_RUN),
                   '--destination', str(DRIVE_BACKUP_RUN)]
    subprocess.run(command, cwd=REPO_DIR, check=True)
else:
    print('Set SYNC_ACTION when validation, restoration, status, or retry is needed.')

# After restore, continue in a fresh process:
# subprocess.run([sys.executable, '-u', 'train_diffcrl.py', '--config', str(formal_path),
#                 '--resume-from', str(LOCAL_RESTORED_RUN),
#                 '--stage-sync-to', str(DRIVE_BACKUP_RUN)], cwd=REPO_DIR, check=True)